# Notebook 02 — Data Cleaning

## Objectives
- Load the combined dataset from Notebook 01.
- Investigate and handle data-quality issues found during Data Understanding:
  missing values, duplicate rows, impossible years, invalid engine sizes,
  impossible fuel-economy figures, and a price sense-check.
- Save a cleaned dataset for feature engineering.

## Inputs
- `outputs/datasets/collection/used_cars.csv` (combined raw data, 99,187 rows).

## Outputs
- `outputs/datasets/cleaned/used_cars_cleaned.csv` (cleaned data).

> Satisfies Merit **7.2** (Data Preparation — a dedicated cleaning notebook that
> states how issues were investigated and handled) under CRISP-DM.

---
## 1. Load the combined data

In [1]:
import os
import pandas as pd

if os.path.basename(os.getcwd()) == "jupyter_notebooks":
    os.chdir(os.path.dirname(os.getcwd()))
print("Working directory:", os.getcwd())

df = pd.read_csv("outputs/datasets/collection/used_cars.csv")
print("Loaded:", df.shape)
df.head()

Working directory: /workspaces/used-car-price-predictor
Loaded: (99187, 10)


,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,manufacturer
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4,Audi
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,Audi
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,Audi
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,Audi
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,Audi


---
## 2. Missing values

We investigate missing values with `isna().sum()`. From Notebook 01 we expect none,
but we confirm here as part of Data Preparation.

In [2]:
df.isna().sum()

model           0
year            0
price           0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
manufacturer    0
dtype: int64

**Finding:** every column is fully populated — there are no `NaN` values to impute
or drop. (Note: some columns use placeholder values such as `engineSize == 0`, which
behave like hidden missing values — handled in section 5.)

---
## 3. Duplicate rows

Scraped listings often contain exact duplicates (the same car captured twice). These
add no information and can bias the model, so we identify and remove them.

In [3]:
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes}")

Exact duplicate rows: 1475


In [4]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows. New shape: {df.shape}")

Removed 1475 duplicate rows. New shape: (97712, 10)


---
## 4. Impossible years

The data was scraped in 2020, so any registration year after 2020 is impossible
(Data Understanding showed a maximum of 2060). We inspect the offending rows, then
remove them.

In [5]:
print("Year values greater than 2020:")
print(df.loc[df["year"] > 2020, "year"].value_counts())

Year values greater than 2020:
year
2060    1
Name: count, dtype: int64


In [6]:
before = len(df)
df = df[df["year"] <= 2020].reset_index(drop=True)
print(f"Removed {before - len(df)} rows with impossible years. New shape: {df.shape}")

Removed 1 rows with impossible years. New shape: (97711, 10)


---
## 5. Invalid engine size

A combustion engine cannot have a size of 0.0 litres. We check which fuel types these
rows belong to (electric cars would legitimately have no engine size, but our pricing
model is built around conventional vehicles), then remove them as invalid/out-of-scope.

In [7]:
print("Rows with engineSize == 0:", (df["engineSize"] == 0).sum())
print("\nTheir fuel types:")
print(df.loc[df["engineSize"] == 0, "fuelType"].value_counts())

Rows with engineSize == 0: 268

Their fuel types:
fuelType
Petrol      158
Diesel       69
Hybrid       38
Electric      2
Other         1
Name: count, dtype: int64


In [8]:
before = len(df)
df = df[df["engineSize"] > 0].reset_index(drop=True)
print(f"Removed {before - len(df)} rows with engineSize 0. New shape: {df.shape}")

Removed 268 rows with engineSize 0. New shape: (97443, 10)


---
## 6. Impossible fuel economy (mpg)

Data Understanding showed `mpg` ranging from 0.3 to 470.8. A value below ~1 mpg is
physically impossible (likely a decimal/typo error), so we inspect and remove those.
Very high values are inspected too — they correspond to plug-in hybrids' official
combined figures, so they are retained for now and revisited during outlier handling
in feature engineering.

In [9]:
print("Rows with mpg < 1:", (df["mpg"] < 1).sum())
print(df.loc[df["mpg"] < 1, "mpg"].value_counts())
print("\nTen highest mpg values:")
print(df["mpg"].sort_values(ascending=False).head(10).to_list())

Rows with mpg < 1: 1
mpg
0.3    1
Name: count, dtype: int64

Ten highest mpg values:
[470.8, 470.8, 470.8, 470.8, 470.8, 470.8, 470.8, 470.8, 256.8, 256.8]


In [10]:
before = len(df)
df = df[df["mpg"] >= 1].reset_index(drop=True)
print(f"Removed {before - len(df)} rows with impossible mpg. New shape: {df.shape}")

Removed 1 rows with impossible mpg. New shape: (97442, 10)


---
## 7. Price sense-check

`price` is our target, so we sense-check its extremes rather than blindly trim them.
We look at the cheapest and most expensive listings to judge whether the range is
plausible.

In [11]:
print(df["price"].describe())
print("\nCheapest 5 listings:")
print(df.nsmallest(5, "price")[["manufacturer", "model", "year", "mileage", "price"]])
print("\nMost expensive 5 listings:")
print(df.nlargest(5, "price")[["manufacturer", "model", "year", "mileage", "price"]])

count     97442.000000
mean      16773.723918
std        9869.933225
min         450.000000
25%        9999.000000
50%       14470.500000
75%       20750.000000
max      159999.000000
Name: price, dtype: float64

Cheapest 5 listings:
      manufacturer   model  year  mileage  price
74936     Vauxhall   Astra  2001   159000    450
82552     Vauxhall   Agila  2003    90000    450
38029         Ford   Focus  2003   177644    495
72436     Vauxhall   Corsa  2002    99842    495
72450     Vauxhall   Corsa  2003    82000    590

Most expensive 5 listings:
      manufacturer      model  year  mileage   price
49719     Mercedes    G Class  2020     1350  159999
53410     Mercedes    G Class  2020     3000  154998
43621     Mercedes   SL CLASS  2011     3000  149948
4743          Audi         R8  2020     2000  145000
52159     Mercedes    A Class  2019      785  140319


**Finding:** the cheapest cars are old, high-mileage vehicles and the most expensive
are recent premium models — the price range is plausible, so no price rows are removed.
Any residual outliers are left for the feature-engineering stage.

---
## 8. Final check and save

In [12]:
print("Final cleaned shape:", df.shape)
df.describe()

Final cleaned shape: (97442, 10)


,year,price,mileage,tax,mpg,engineSize
count,97442.000000,97442.000000,97442.000000,97442.000000,97442.000000,97442.000000
mean,2017.066727,16773.723918,23224.786776,120.165842,55.056856,1.669502
std,2.113094,9869.933225,21069.221991,63.326000,14.119089,0.552466
min,1970.000000,450.000000,1.000000,0.000000,1.100000,0.600000
25%,2016.000000,9999.000000,7672.000000,125.000000,47.100000,1.200000
50%,2017.000000,14470.500000,17679.000000,145.000000,54.300000,1.600000
75%,2019.000000,20750.000000,32500.000000,145.000000,62.800000,2.000000
max,2020.000000,159999.000000,323000.000000,580.000000,470.800000,6.600000


In [13]:
out_dir = "outputs/datasets/cleaned"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "used_cars_cleaned.csv")
df.to_csv(out_path, index=False)
print("Saved cleaned dataset:", out_path, "| rows:", len(df), "| columns:", df.shape[1])

Saved cleaned dataset: outputs/datasets/cleaned/used_cars_cleaned.csv | rows: 97442 | columns: 10


## 9. Conclusions and next steps
Cleaning removed **1,745 rows (1.8%)**, leaving **97,442 clean records**:

- **1,475** exact duplicate listings.
- **1** row with an impossible year (2060).
- **268** rows with `engineSize` of 0 — mostly petrol/diesel cars where the size was
  clearly missing (only 2 were genuinely electric), so removal is safe and keeps the
  dataset focused on conventional vehicles.
- **1** row with an impossible `mpg` (0.3).

The cleaned ranges are now sensible: `year` 1970–2020, `engineSize` 0.6–6.6 L,
`price` £450–£159,999, and `mpg` 1.1–470.8 (the very high figures are plug-in hybrids'
official combined values, revisited during outlier handling). No missing values remain.

Next: **Notebook 03 — Correlation Study & Hypothesis Validation** (answers Business
Requirement 1 and validates the project hypotheses).